<a href="https://colab.research.google.com/github/Fdiannz/BT4222-Project-Group-4--MIND/blob/main/step4_advanced_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# step4_advanced_model: Knowledge-aware LSTM–Attention Recommender (KLAR)

**Owner:** P5

**Purpose:** This is the group's advanced model, **KLAR (Knowledge-aware LSTM–Attention Recommender)**. For each impression (one visit where a user is shown a slate of articles), it ranks the candidate articles by matching each article's **content** (title words, category and knowledge-graph entities) against the user's **click history, read in order**. Articles are represented only by content available at publication time, never by an ID embedding or click counts, so newly published (**cold**) articles can be scored from the moment they appear.

**Inputs**
- `data/processed/` from step 1: `news_metadata` (article content), `fit_*` (training, 9–13 Nov), `tune_*` (tuning, 14 Nov), `eval_*` (test, 15 Nov)
- `entity_embedding.vec`: MIND's pre-trained **TransE** vectors for Wikidata entities (people, places, organisations)
- **word2vec-google-news-300**: pre-trained word vectors, downloaded once and cached

**Outputs (shared)**
- **Scores:** `data/scores/step4_klar_tune.parquet`, `step4_klar_eval.parquet` and `step4_klar_train_sample.parquet`. Each file has one row per article shown in an impression, with columns `impression_id`, `news_id` and `score`. The score is the raw dot product, and a higher score means the user is more likely to click. Step 5 uses these files for MMR re-ranking and step 6 uses them for diagnostics.
- **News representations:** `data/embeddings/step4_klar_news_vectors.npy` holds the model's 256-dimensional vector for every article. `step4_klar_news_ids.parquet` lists the matching article IDs in the same row order. Step 5 uses them to measure how similar two articles are.
- **Run files:** `outputs/step4_klar/` holds the saved model checkpoints, the training log of each epoch and the result tables (overall, cold vs warm, by history length and ablations) used in the report.


### Architecture
1. **News representation:** one 256-d vector per article, used for both history and candidate articles; no attention on this side.
   - **Title embedding:** up to 30 title words → **word2vec** vectors (fine-tuned during training) → average of the real words → linear layer (300 → 256).
   - **Category embedding:** learned embeddings for the category (e.g. *sports*) and the category_subcategory pair (e.g. *sports_nfl*), 100 each.
   - **Entity embedding:** up to 10 entities from the title and abstract. Each entity combines its fixed **TransE** vector (its position in the Wikidata knowledge graph), a learned **entity type** embedding (person, organisation, place, …), its **linking confidence** and an **in-title flag**. These go through a linear layer, and the entities are combined by a **confidence-weighted average**. A **has-entity flag** marks articles with none.
   - **Fusion:** all parts are concatenated and a linear layer maps them to the final 256-d news representation.
2. **User encoder:** one 256-d **user vector** per impression, built from the news representations of the user's 50 most recent clicks (oldest → newest):
   - **LSTM:** reads the clicks in order, so each hidden state summarises the history up to that click.
   - **Multi-head self-attention** (16 heads): lets every click relate directly to every other click, even far apart.
   - **Additive attention:** combines the 50 hidden states into one user vector.

   Users with no history get a learned **"empty user" vector**.
3. **Click score** = **dot product** of the user vector and the candidate's news representation.

**Training:** each click is paired with 4 unclicked articles from the **same impression** (negative sampling). The model scores all 5, and **softmax cross-entropy** pushes the clicked one to the top.

### Design rationale

| Component | Why it is used |
|---|---|
| Content only, no user or article ID embeddings | An ID embedding is learned from clicks, so a new article or user would have nothing to go on. Content exists from the moment an article is published, which is what lets KLAR score cold articles. |
| Word2vec title embedding (fine-tuned), averaged over words | Pre-trained news vectors give every word a sensible starting meaning, even rare ones, and fine-tuning adapts them to what readers click. Averaging keeps the article side simple and fast.  |
| Category and subcategory embeddings | Readers have stable topic interests, and every article has a category, even when its title is vague. |
| TransE entity embeddings | Entities tell the model who and what a story is about, even when two titles share no words. TransE vectors come from the Wikidata knowledge graph, not from clicks, so a new article about a known person or team is placed near related articles straight away. |
| LSTM over the click history | Clicks happen in an order. The LSTM reads them in sequence so recent interests can differ from older ones. |
| Multi-head self-attention + additive attention | Self-attention lets any two clicks relate directly, even far apart in the history. Additive attention then combines all clicks into one user vector. |
| Negatives from the same impression | Each click is trained against 4 articles the user was shown but skipped, so the model learns to rank within a real slate, the same task it is evaluated on. |
| Separate tune and test days | Settings and the best epoch are chosen on 14 Nov, and the model is tested once on 15 Nov, so the test result is never used for choices. |

### Inputs left out on purpose
- **Popularity (past clicks or click rate):** it favours articles that have already been shown many times, which works against new articles, the cases KLAR is built for.
- **Time of day / day of week:** it is the same for every article in an impression, so it cannot help rank them. The tune and test days (Thu 14 and Fri 15 Nov) are also weekdays that never appear in the training days (Sat 9 – Wed 13 Nov).

## 0. Setup

In [1]:
# --- Setup: run this first, every session ---
from google.colab import drive
drive.mount('/content/drive')

# Fresh copy of the team repo (shared config.py and utils/) on every run
!rm -rf /content/BT4222-Project-Group-4--MIND && git clone -q https://github.com/Fdiannz/BT4222-Project-Group-4--MIND.git /content/BT4222-Project-Group-4--MIND
import sys; sys.path.insert(0, '/content/BT4222-Project-Group-4--MIND')
!pip -q install gensim

import config
print("Data folder:", config.PROCESSED_DIR)

Mounted at /content/drive
Data folder: /content/drive/MyDrive/BT4222_Project/data/processed


## 1. Settings

- **`CFG`** holds every model and training setting (sizes, learning rate, epochs and so on) in one place.
- **`COLS`** links the column names used in this notebook to the column names in step 1's files. If step 1 renames a column, only `COLS` needs to change.
- **`RUN`** is the name the model is saved under on Drive.
- **`RESUME = True`:** if a run with that name has already finished, the notebook loads it instead of training again. If it stopped halfway, training continues from the last saved epoch.
  Change `RUN` whenever  `CFG` is changed (e.g. `"klar_v1"` → `"klar_v2"`). Otherwise the notebook will load the old model trained with the old settings.
- **`RUN_ABLATIONS`:** set it to `False` to skip the ablations (section 14).


In [2]:
import os, re, json, time, math, gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from utils.metrics import evaluate, evaluate_by_segment, label_impression_segment

# ---- folders (all from the shared config.py) ----
PROC = config.PROCESSED_DIR
DATA_DIR = getattr(config, "DATA_DIR", os.path.dirname(PROC))
OUTPUT_DIR = getattr(config, "OUTPUT_DIR", os.path.join(os.path.dirname(DATA_DIR), "outputs"))
SCORE_DIR = getattr(config, "SCORES_DIR", os.path.join(DATA_DIR, "scores"))   # shared score files
EMB_DIR = os.path.join(DATA_DIR, "embeddings")                                  # shared article vectors
RUN_DIR = os.path.join(OUTPUT_DIR, "step4_klar")                                # checkpoints, logs, tables
for d in (SCORE_DIR, EMB_DIR, RUN_DIR):
    os.makedirs(d, exist_ok=True)
PREFIX = "step4_klar"                                                           # shared file names

FILES = {
    "news": "news_metadata.parquet",
    "fit_imp": "fit_impressions.parquet",   "fit_cand": "fit_candidates_sampled.parquet",
    "tune_imp": "tune_impressions.parquet", "tune_cand": "tune_candidates.parquet",
    "eval_imp": "eval_impressions.parquet", "eval_cand": "eval_candidates.parquet",
}

# name used here -> column name in step 1's files
COLS = {
    "impression_id": "impression_id",
    "news_id": "news_id",
    "label": "label",
    "is_cold": "cold_article",                  # 1 if the article had < 10 displays on earlier days
    "prior_count": "prior_exposure_count",      # displays on earlier days (used only for checks)
    "time": "time",
    "history": "history",
    "impressions": "impressions",               # full slate string 'N1-0 N2-1 ...' in *_impressions
    "title": "title",
    "category": "category",
    "subcategory": "subcategory",
    "title_entities": "title_entities",         # JSON list of linked Wikidata entities
    "abstract_entities": "abstract_entities",
}

CFG = dict(
    # data
    title_len=30,             # title words kept per article
    max_hist=50,              # most recent history clicks per user
    neg_k=4,                  # negatives per click, all from the same impression
    n_train_clicks=1_500_000, # random subset of fit clicks (compute budget)
    n_tune_imps=20_000,       # fixed tune subset scored after each epoch to pick the best epoch
    n_train_slate_imps=20_000,# training impressions re-scored on full slates (train vs test comparison)
    w2v_limit=500_000,        # most frequent word2vec words loaded (saves RAM)
    min_subcat_articles=10,   # fewer training articles -> subcategory shares its category's "other" slot
    max_entities=10,          # entities per article (title first)
    # model
    d_model=256,              # news representation and user vector size
    cat_dim=100,              # each category embedding
    ent_dim=100,              # entity embedding
    ent_type_dim=20,          # entity type embedding
    n_heads=16,               # multi-head self-attention across clicked articles (256 / 16 = 16 dims each)
    query_dim=200,            # additive attention query size
    dropout=0.2,
    use_category=True,        # ablation switches
    use_entities=True,
    use_lstm=True,
    # training
    lr=1e-4, batch_size=64, epochs=6,
    seed=getattr(config, "SEED", 42),
)
RUN = "klar_v1"               # checkpoint name of the main model
USE_W2V = True                # False = random word vectors (quick dry run only)
RESUME = True                 # load a finished run / continue an interrupted one
RUN_ABLATIONS = True          # False = skip section 14

# MIND's TransE entity vectors (one file per official split; all from the same knowledge graph)
RAW_DIR = getattr(config, "RAW_DIR", os.path.join(DATA_DIR, "raw"))
ENTITY_FILES = [os.path.join(getattr(config, k, os.path.join(RAW_DIR, d)), "entity_embedding.vec")
                for k, d in [("TRAIN_DIR", "MINDlarge_train"), ("DEV_DIR", "MINDlarge_dev"), ("TEST_DIR", "MINDlarge_test")]]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"     # mixed precision on GPU: faster, same results up to rounding

def seed_everything(seed):
    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CFG["seed"])
print("Device:", DEVICE, "| run:", RUN)

Device: cuda | run: klar_v1


## 2. Load the data and check for leakage

Only the columns this notebook needs are loaded. Four checks run before training to make sure no future information leaks into the model:

1. **Days don't overlap:** every training impression (9–13 Nov) comes before every tuning impression (14 Nov), which comes before every test impression (15 Nov).
2. **Cold labels only look backwards:** an article's "prior display count" (used to label it cold or warm) must stay the same within a day, which shows it only counts earlier days.
3. **Every candidate has content:** each article shown in an impression must exist in `news_metadata`, so it can be represented.
4. **History is from the past:** MIND builds each user's history from clicks before 9 Nov. The notebook prints the share of clicked articles that already appear in that user's history. A small share is normal (readers re-opening an article). A large share would mean the history contains the click being predicted.



In [3]:
def load(name, want):
    # Read only the wanted columns that exist in the file, renamed to the names used here.
    path = os.path.join(PROC, FILES[name])
    have = pq.read_schema(path).names
    cols = [COLS[c] for c in want if COLS[c] in have]
    df = pd.read_parquet(path, columns=cols).rename(columns={COLS[c]: c for c in want})
    print(f"{name:10s} {len(df):>12,} rows | columns used: {list(df.columns)}")
    return df, have

NEED = {
    "news": ["news_id", "title", "category", "subcategory", "title_entities", "abstract_entities"],
    "fit_imp": ["impression_id", "history"], "fit_cand": ["impression_id", "news_id", "label", "time"],
    "tune_imp": ["impression_id", "history"], "tune_cand": ["impression_id", "news_id", "label", "is_cold", "time", "prior_count"],
    "eval_imp": ["impression_id", "history"], "eval_cand": ["impression_id", "news_id", "label", "is_cold", "time", "prior_count"],
}
data, schemas = {}, {}
for name in FILES:
    data[name], schemas[name] = load(name, NEED[name])
missing = {n: [c for c in NEED[n] if c not in data[n].columns] for n in NEED}
missing = {n: c for n, c in missing.items() if c}
assert not missing, f"Missing columns {missing}: update COLS in section 1."

news = data["news"].drop_duplicates("news_id").reset_index(drop=True)
fit_cand, fit_imp = data["fit_cand"], data["fit_imp"]
tune_cand, tune_imp = data["tune_cand"], data["tune_imp"]
eval_cand, eval_imp = data["eval_cand"], data["eval_imp"]
del data

# ---- check 1: no time overlap between fit, tune and test ----
spans = {n: (d.time.min(), d.time.max()) for n, d in [("fit", fit_cand), ("tune", tune_cand), ("eval", eval_cand)]}
for n, (a, b) in spans.items():
    print(f"{n:5s} {a} -> {b}")
assert spans["fit"][1] < spans["tune"][0] and spans["tune"][1] < spans["eval"][0], "splits overlap in time"
print("check 1 passed: fit < tune < test in time")

# ---- check 2: cold labels are frozen within each day ----
for n, d in [("tune", tune_cand), ("eval", eval_cand)]:
    frozen = (d.groupby([d["news_id"], d["time"].dt.date])["prior_count"].nunique() == 1).mean()
    assert frozen == 1, f"{n}: prior display counts change within a day"
print("check 2 passed: prior display counts use earlier days only")

# ---- check 3: every candidate has article content ----
known = set(news["news_id"])
for n, d in [("fit", fit_cand), ("tune", tune_cand), ("eval", eval_cand)]:
    miss = (~d["news_id"].isin(known)).sum()
    assert miss == 0, f"{miss:,} {n} candidates are missing from news_metadata"
print("check 3 passed: all candidates have article content")

# ---- check 4: clicked articles already in the same impression's history ----
def clicked_in_history(name):
    hits = total = 0
    pf = pq.ParquetFile(os.path.join(PROC, FILES[name]))
    for b in pf.iter_batches(batch_size=200_000, columns=[COLS["history"], COLS["impressions"]]):
        for h, s in zip(*[c.to_pylist() for c in b.columns]):
            hist = set(h.split()) if isinstance(h, str) else set()
            for item in s.split():
                if item.endswith("-1"):
                    total += 1
                    hits += item[:-2] in hist
    return hits, total

for name in ("fit_imp", "tune_imp", "eval_imp"):
    hits, total = clicked_in_history(name)
    print(f"check 4 {name[:-4]:4s}: {hits:,} of {total:,} clicks ({hits / total:.2%}) are already in the user's own history")

for n, d in [("fit (sampled)", fit_cand), ("tune", tune_cand), ("eval", eval_cand)]:
    extra = f", cold share {d.is_cold.mean():.3f}" if "is_cold" in d else ""
    print(f"{n:14s}: {d.impression_id.nunique():>9,} impressions, click rate {d.label.mean():.3f}{extra}")

# keep only the columns used from here on (the full tune and test days have ~30M candidate rows)
fit_cand = fit_cand[["impression_id", "news_id", "label"]].astype({"label": "int8"})
tune_cand = tune_cand[["impression_id", "news_id", "label", "is_cold"]].astype({"label": "int8", "is_cold": "int8"})
eval_cand = eval_cand[["impression_id", "news_id", "label", "is_cold"]].astype({"label": "int8", "is_cold": "int8"})
gc.collect()

news            104,151 rows | columns used: ['news_id', 'title', 'category', 'subcategory', 'title_entities', 'abstract_entities']
fit_imp       1,801,231 rows | columns used: ['impression_id', 'history']
fit_cand     12,894,210 rows | columns used: ['impression_id', 'news_id', 'label', 'time']
tune_imp        431,517 rows | columns used: ['impression_id', 'history']
tune_cand    17,400,106 rows | columns used: ['impression_id', 'news_id', 'label', 'is_cold', 'time', 'prior_count']
eval_imp        376,471 rows | columns used: ['impression_id', 'history']
eval_cand    14,085,557 rows | columns used: ['impression_id', 'news_id', 'label', 'is_cold', 'time', 'prior_count']
fit   2019-11-09 00:00:00 -> 2019-11-13 23:59:57
tune  2019-11-14 00:00:00 -> 2019-11-14 23:59:59
eval  2019-11-15 00:00:00 -> 2019-11-15 23:59:43
check 1 passed: fit < tune < test in time
check 2 passed: prior display counts use earlier days only
check 3 passed: all candidates have article content
check 4 fit : 16,216 

124

## 3. Tokenise titles into word IDs

Each title is lower-cased, split into words, and each word is replaced by its ID number from the vocabulary. Every title is then cut or padded with 0s to exactly 30 IDs, so all titles stack into one matrix.

Example: *"Patriots beat Jets, Brady's 3 TDs"* → `[812, 95, 1460, 1, 37, 2201, 0, 0, …, 0]`

- **Vocabulary:** built from the words in all titles. Words that appear only once become `<unk>` (ID 1), and padding is ID 0. Only text is used, no clicks or labels. A title is known as soon as the article is published, so including test-day titles is not leakage.
- **Row numbers:** row 0 is an empty "padding article" used to fill short histories. So article row *r* is `news.loc[r - 1]`.


In [4]:
TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?")
def tokenize(text):
    return TOKEN_RE.findall(str(text).lower()) if isinstance(text, str) else []

# news_id -> article row (row 0 = padding article)
NEWS_ROW = pd.Series(np.arange(1, len(news) + 1), index=news["news_id"].to_numpy())

def build_vocab(texts):
    counts = {}
    for t in texts:
        for w in tokenize(t):
            counts[w] = counts.get(w, 0) + 1
    words = sorted(w for w, c in counts.items() if c >= 2)        # rare words -> <unk>
    return {"<pad>": 0, "<unk>": 1, **{w: i + 2 for i, w in enumerate(words)}}

def build_tokens(texts, length, vocab):
    tok = np.zeros((len(texts) + 1, length), dtype=np.int64)
    for i, t in enumerate(texts):
        ids = [vocab.get(w, 1) for w in tokenize(t)][:length]
        tok[i + 1, :len(ids)] = ids
    return tok

VOCAB = build_vocab(news["title"].fillna(""))
TOKENS = build_tokens(news["title"].fillna("").tolist(), CFG["title_len"], VOCAB)
title_len = (TOKENS[1:] > 0).sum(1)
print(f"Vocabulary: {len(VOCAB):,} words | articles: {len(news):,} | "
      f"title words kept: median {np.median(title_len):.0f}, empty titles {np.mean(title_len == 0):.2%}")
print("Example:", news.loc[0, "title"], "->", TOKENS[1, :12])

Vocabulary: 27,554 words | articles: 104,151 | title words kept: median 11, empty titles 0.00%
Example: The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By -> [24652  3550 19570  8365 19144  4750  1540 19144 18383 24111  4060     0]


## 4. Prepare category and entity inputs for each article

This section turns each article's category labels and linked entities into numbers the model can look up. Each article gets one row, in the same order as the title IDs. Only article content is used, no clicks or labels.

**Category IDs:** each article gets two IDs, one for its category (e.g. *sports*) and one for its category_subcategory (e.g. *sports_basketball_nba*). The model learns an embedding for each ID.
- Only categories seen in the training articles (9–13 Nov) get an ID, so every embedding is actually trained.
- A subcategory with fewer than 10 training articles is grouped into `<category>_other` (e.g. *sports_other*).
- A category never seen in training gets ID 0, a fixed zero vector.

**Entities:** the people, places and organisations linked in each article's title and abstract (`title_entities`, `abstract_entities`). Each article keeps up to 10, title entities first, with no repeats. For each entity we store:
- its **TransE vector** (100 numbers from MIND's `entity_embedding.vec`), which describes its position in the Wikidata knowledge graph
- its **type** (person, organisation, place, …) as an ID (only types seen in training articles get one)
- its **linking confidence** (0–1), how sure MIND's entity linker is about the match, also used to weight the average
- an **in-title flag**: 1 if it appears in the title, 0 if only in the abstract

Entities without a TransE vector are dropped. `HAS_ENT` marks articles that have at least one usable entity. TransE vectors are kept fixed. The model learns only the entity type embedding and the linear layer that combines each entity’s inputs.




In [5]:
# ---------- category and category_subcategory IDs (slots only for values seen in training) ----------
def build_category_ids(min_articles):
    cat = news["category"].fillna("").str.lower()
    sub = cat + "_" + news["subcategory"].fillna("").str.lower()
    in_fit = news["news_id"].isin(fit_cand["news_id"]).to_numpy()      # articles in the training data
    n_sub = sub[in_fit].value_counts()
    sub = sub.where(sub.map(n_sub).fillna(0).to_numpy() >= min_articles, cat + "_other")

    def to_ids(labels):
        vocab = {v: i + 1 for i, v in enumerate(sorted(labels[in_fit].unique()))}   # 0 = unseen -> zero vector
        return np.r_[0, labels.map(vocab).fillna(0).to_numpy(dtype=np.int64)], vocab
    (cat_ids, cat_vocab), (sub_ids, sub_vocab) = to_ids(cat), to_ids(sub)
    return cat_ids, sub_ids, cat_vocab, sub_vocab

CAT_IDS, SUB_IDS, CAT_VOCAB, SUB_VOCAB = build_category_ids(CFG["min_subcat_articles"])
print(f"Categories: {len(CAT_VOCAB)} | category_subcategory slots: {len(SUB_VOCAB)} "
      f"(incl. {sum(k.endswith('_other') for k in SUB_VOCAB)} 'other' slots) | "
      f"articles with an unseen category: {(CAT_IDS[1:] == 0).mean():.2%}")

# ---------- entities: TransE vector + type + confidence + in-title flag ----------
def parse_entities(s):
    # JSON list of entity objects -> [(WikidataId, type, confidence, first offset)]; malformed or empty -> []
    if not isinstance(s, str) or not s.startswith("["):
        return []
    try:
        return [(e["WikidataId"], str(e.get("Type", "")), float(e.get("Confidence", 1.0)),
                 min(e.get("OccurrenceOffsets") or [0])) for e in json.loads(s) if e.get("WikidataId")]
    except (ValueError, TypeError, KeyError):
        return []

def article_entities():
    # one list per article: [(qid, type, confidence, in_title)], title entities first, each in order of mention
    out = []
    for t_s, a_s in zip(news["title_entities"], news["abstract_entities"]):
        seen, ents = set(), []
        for s, in_title in ((t_s, 1.0), (a_s, 0.0)):
            for qid, typ, conf, _ in sorted(parse_entities(s), key=lambda e: e[3]):
                if qid not in seen:
                    seen.add(qid)
                    ents.append((qid, typ, conf, in_title))
        out.append(ents)
    return out

def load_transe(files, wanted):
    vecs = {}
    for path in files:
        if not os.path.exists(path):
            print("  not found (skipped):", path)
            continue
        with open(path) as f:
            for line in f:
                parts = line.split()
                if parts and parts[0] in wanted and parts[0] not in vecs:
                    vecs[parts[0]] = np.asarray(parts[1:], dtype=np.float32)
        print(f"  read {path}: {len(vecs):,} needed entities so far")
    return vecs

def build_entity_inputs(article_ents, transe, max_ent):
    # one row per article (row 0 = padding), max_ent entity slots each; slot value 0 = empty
    qids = sorted(transe)
    table = np.zeros((len(qids) + 1, len(transe[qids[0]])), dtype=np.float32)     # row 0 = empty slot
    table[1:] = np.stack([transe[q] for q in qids])
    q_row = {q: i + 1 for i, q in enumerate(qids)}
    in_fit = news["news_id"].isin(fit_cand["news_id"]).to_numpy()
    type_id = {t: i + 1 for i, t in enumerate(sorted({t for ents, f in zip(article_ents, in_fit) if f
                                                      for _, t, _, _ in ents}))}  # 0 = unseen type
    n = len(article_ents) + 1
    idx, typ = np.zeros((n, max_ent), dtype=np.int64), np.zeros((n, max_ent), dtype=np.int64)
    conf, title = np.zeros((n, max_ent), dtype=np.float32), np.zeros((n, max_ent), dtype=np.float32)
    for i, ents in enumerate(article_ents):
        for j, (q, t, c, tt) in enumerate([e for e in ents if e[0] in q_row][:max_ent]):
            idx[i + 1, j], typ[i + 1, j], conf[i + 1, j], title[i + 1, j] = q_row[q], type_id.get(t, 0), c, tt
    return {"ent_table": table, "ent_idx": idx, "ent_type": typ, "ent_conf": conf, "ent_title": title,
            "n_types": len(type_id)}

ARTICLE_ENTS = article_entities()
TRANSE = load_transe(ENTITY_FILES, {e[0] for ents in ARTICLE_ENTS for e in ents})
assert TRANSE, "entity_embedding.vec not found: put it in data/raw/MINDlarge_train/ (and the dev/test folders)"
ENT_INPUTS = build_entity_inputs(ARTICLE_ENTS, TRANSE, CFG["max_entities"])
HAS_ENT = (ENT_INPUTS["ent_idx"][:, 0] > 0).astype(np.float32)
n_ent = (ENT_INPUTS["ent_idx"][1:] > 0).sum(1)
print(f"Entities: {len(TRANSE):,} with a TransE vector ({ENT_INPUTS['ent_table'].shape[1]}-d), "
      f"{ENT_INPUTS['n_types']} entity types | articles with a usable entity: {HAS_ENT[1:].mean():.1%} | "
      f"entities per article: median {np.median(n_ent):.0f}, more than {CFG['max_entities']}: "
      f"{np.mean([len(e) > CFG['max_entities'] for e in ARTICLE_ENTS]):.2%} of articles")
rows = NEWS_ROW.reindex(eval_cand["news_id"].to_numpy()).to_numpy()
for flag, name in [(1, "cold"), (0, "warm")]:
    print(f"  test {name} candidates with an entity: {HAS_ENT[rows[eval_cand.is_cold.to_numpy() == flag]].mean():.1%}")
del TRANSE, ARTICLE_ENTS, rows
news = news.drop(columns=["title_entities", "abstract_entities"])
gc.collect()

FEATURES = {"cat_ids": CAT_IDS, "sub_ids": SUB_IDS, "n_cat": len(CAT_VOCAB), "n_sub": len(SUB_VOCAB),
            "entities": ENT_INPUTS, "has_ent": HAS_ENT}

Categories: 16 | category_subcategory slots: 139 (incl. 15 'other' slots) | articles with an unseen category: 0.00%
  read /content/drive/MyDrive/BT4222_Project/data/raw/MINDlarge_train/entity_embedding.vec: 42,007 needed entities so far
  read /content/drive/MyDrive/BT4222_Project/data/raw/MINDlarge_dev/entity_embedding.vec: 42,628 needed entities so far
  read /content/drive/MyDrive/BT4222_Project/data/raw/MINDlarge_test/entity_embedding.vec: 42,628 needed entities so far
Entities: 42,628 with a TransE vector (100-d), 22 entity types | articles with a usable entity: 87.9% | entities per article: median 2, more than 10: 0.22% of articles
  test cold candidates with an entity: 83.5%
  test warm candidates with an entity: 78.0%


## 5. Initialise the title word embeddings with word2vec

This gives each word ID from section 3 its starting vector, for the **title embedding** part of the news representation.

- The notebook builds a table with one row of 300 numbers per vocabulary word (about 27,500 words × 300).
- Each row starts from that word's pre-trained **word2vec** vector (word2vec-google-news-300, trained on Google News, the same kind of text as MIND). So words with similar meanings start close together, e.g. *patriots* near *jets*.
- Words not in word2vec (about 12%: names, rare words, possessives like *brady's*) start from small random vectors.
- During training all rows are **fine-tuned**, so the vectors adapt to what readers actually click.

The word2vec download (about 1.6 GB) happens once, and the finished table is saved to Drive for later runs.

In [6]:
EMB_DIM = 300
def build_word_vectors(vocab):
    cache = os.path.join(RUN_DIR, f"word2vec_init_{len(vocab)}.npy")
    if os.path.exists(cache):
        print("Loaded cached word vectors:", cache)
        return np.load(cache)
    rng = np.random.default_rng(CFG["seed"])
    emb = rng.normal(0, 0.1, (len(vocab), EMB_DIM)).astype(np.float32)
    emb[0] = 0.0                                          # padding word
    if not USE_W2V:
        print("USE_W2V=False: random word vectors (dry run only)")
        return emb
    import gensim.downloader as api
    from gensim.models import KeyedVectors
    kv = KeyedVectors.load_word2vec_format(api.load("word2vec-google-news-300", return_path=True),
                                           binary=True, limit=CFG["w2v_limit"])
    hits = 0
    for w, i in vocab.items():
        if i < 2:
            continue
        for form in (w, w.capitalize(), w.upper()):        # word2vec is case-sensitive
            if form in kv.key_to_index:
                emb[i] = kv[form]; hits += 1
                break
    print(f"word2vec coverage: {hits:,}/{len(vocab) - 2:,} words ({hits / (len(vocab) - 2):.1%})")
    np.save(cache, emb)
    return emb

WORD_EMB = build_word_vectors(VOCAB)

[==================================================] 100.0% 1662.8/1662.8MB downloaded
word2vec coverage: 24,177/27,552 words (87.8%)


## 6. Build each user's click history

For each impression, the user's past clicks (from MIND's history, all before 9 Nov) become a list of 50 article row numbers, which the user encoder reads.

- **Most recent 50 kept:** a user with more than 50 past clicks keeps only the latest 50.
- **Oldest → newest:** the order is kept, because the LSTM reads the clicks in sequence.
- **Padding at the end:** a user with fewer than 50 clicks has their clicks first, then 0s (the empty padding article) to fill the rest.
- **Unknown articles dropped:** a history article not found in `news_metadata` is removed rather than left as a gap, so the real clicks stay together at the start.

Example: a user with 3 past clicks → `[17, 2044, 391, 0, 0, …, 0]` (50 numbers)

In [7]:
def parse_history(h):
    return h.split() if isinstance(h, str) else []

def build_history(imp_df, impression_ids=None, max_hist=CFG["max_hist"]):
    # Returns (H, index): H[n_imp, max_hist] article rows, oldest -> newest, padding after; index: impression_id -> row of H
    df = imp_df if impression_ids is None else imp_df[imp_df.impression_id.isin(impression_ids)]
    df = df.drop_duplicates("impression_id")
    H = np.zeros((len(df), max_hist), dtype=np.int64)
    for i, h in enumerate(df["history"].to_numpy()):
        ids = parse_history(h)
        if ids:
            rows = NEWS_ROW.reindex(ids).dropna().to_numpy(dtype=np.int64)[-max_hist:]   # most recent kept, in order
            H[i, :len(rows)] = rows
    return H, pd.Series(np.arange(len(df)), index=df["impression_id"].to_numpy())

tune_H, tune_idx = build_history(tune_imp)
eval_H, eval_idx = build_history(eval_imp)
for name, H in [("tune", tune_H), ("eval", eval_H)]:
    n = (H > 0).sum(1)
    print(f"{name}: {len(H):,} impressions | history length median {np.median(n):.0f}, empty {np.mean(n == 0):.2%}")

tune: 431,517 impressions | history length median 20, empty 2.64%
eval: 376,471 impressions | history length median 19, empty 2.99%


## 7. Build training examples: 1 click + 4 negatives from the same impression

Each training example is one click plus 4 articles the same user was shown in the same impression but did not click (**negatives**). Every negative was really shown and skipped, so it is a genuine "no".

- **Random 1.5M clicks:** a random 1.5M of the ~2.71M clicks in `fit_candidates_sampled` (9–13 Nov) are used, to fit the compute budget.
- **Different negatives per click:** if an impression has 2 clicks, each gets its own 4 negatives. If an impression has fewer than 4 unclicked articles, some are reused (the share is printed, about 10%).
- **Click in column 0:** each example is stored as `[click, neg1, neg2, neg3, neg4]`, so the target is always position 0. The order doesn't bias the model, because each article is scored on its own and its position never enters the score.
- **Shared histories:** `train_H_rows` links each example to its impression's history (from section 6), so a history is stored once even when the impression has several clicks.

Example: `[N5521 (clicked), N88, N3120, N774, N9031]`. The model should give N5521 the highest of the 5 scores.

In [8]:
def build_train_samples(cand, n_clicks, k, seed):
    # Returns (impression_id per example, C[n, 1 + k] article rows with the click in column 0).
    rng = np.random.default_rng(seed)
    cand = cand.assign(row=NEWS_ROW.reindex(cand["news_id"].to_numpy()).fillna(0).to_numpy(dtype=np.int64))
    pos = cand[cand.label == 1]
    if n_clicks and n_clicks < len(pos):
        pos = pos.sample(n=n_clicks, random_state=seed)                            # random click subset
    neg = cand[(cand.label == 0) & cand.impression_id.isin(pos.impression_id)]
    neg = neg.assign(r=rng.random(len(neg))).sort_values(["impression_id", "r"])   # shuffle within impression
    imp_ids, start, count = np.unique(neg.impression_id.to_numpy(), return_index=True, return_counts=True)
    where = pd.Series(np.arange(len(imp_ids)), index=imp_ids)
    pos = pos[pos.impression_id.isin(imp_ids)]                                     # need at least one negative
    g = where.reindex(pos.impression_id.to_numpy()).to_numpy()
    j = pos.groupby("impression_id").cumcount().to_numpy()                         # j-th click of its impression
    offs = (j[:, None] * k + np.arange(k)[None, :]) % count[g][:, None]            # different negatives per click
    neg_rows = neg["row"].to_numpy()[start[g][:, None] + offs]
    return pos["impression_id"].to_numpy(), np.concatenate([pos["row"].to_numpy()[:, None], neg_rows], axis=1)

def training_set(n_clicks):
    imp_ids, C = build_train_samples(fit_cand, n_clicks, CFG["neg_k"], CFG["seed"])
    H, idx = build_history(fit_imp, impression_ids=np.unique(imp_ids))
    H_rows = idx.reindex(imp_ids).to_numpy()
    assert not np.isnan(H_rows.astype(float)).any(), "some training impressions have no history row"
    return imp_ids, C, H, H_rows.astype(np.int64)

train_imp_ids, train_C, fit_H, train_H_rows = training_set(CFG["n_train_clicks"])
reused = np.mean([len(set(r[1:])) < CFG["neg_k"] for r in train_C[:200_000]])
print(f"Training examples: {len(train_C):,} clicks x {train_C.shape[1]} candidates from "
      f"{len(np.unique(train_imp_ids)):,} impressions | examples with a reused negative: {reused:.1%}")

Training examples: 1,500,000 clicks x 5 candidates from 1,146,261 impressions | examples with a reused negative: 10.0%


## 8. Define the KLAR model

The model is built from small parts, each a PyTorch class:

**News side (one vector per article)**
- **`TitleEmbedding`:** looks up the word2vec vector of each title word, averages the real words (ignoring padding), applies dropout, then a linear layer turns 300 numbers into 256.
- **`EntityEmbedding`:** for each entity, joins its fixed TransE vector, learned type embedding, linking confidence and in-title flag, applies dropout, then a linear layer turns them into 100 numbers. The article's entities are then averaged, weighted by confidence. Articles with no entities get a vector of zeros.
- **`KLAR.encode_news`:** builds the **news representation**. It joins the title embedding, category and subcategory embeddings, entity embedding and has-entity flag, then one linear layer maps them to 256 numbers. History and candidate articles both go through this same function.

**User side (one vector per impression)**
- **`UserEncoder`:** takes the news representations of the user's past clicks (oldest → newest):
  1. **LSTM** reads them in order (padding is skipped)
  2. **Multi-head self-attention** lets each click relate to every other click (padding is ignored)
  3. **Dropout**, then **additive attention** combines them into one 256-number user vector

  Users with no history get a learned **"empty user" vector**. For the ablation, `use_lstm=False` skips step 1.
- **`AdditiveAttention`:** gives each item a score with a learned query, `q · tanh(W·h + b)`, turns the scores into weights with softmax, and returns the weighted sum.

**Scoring**
- **`KLAR.forward`:** encodes each distinct article in the batch once (to save time), builds the user vectors, and returns each candidate's **dot-product score**. The scores are left raw, because `F.cross_entropy` applies the softmax itself, which is more numerically stable.



In [9]:
class AdditiveAttention(nn.Module):
    def __init__(self, dim, query_dim):
        super().__init__()
        self.proj = nn.Linear(dim, query_dim)
        self.query = nn.Linear(query_dim, 1, bias=False)

    def forward(self, x, mask):                       # x [N, L, d], mask [N, L] True = real item
        a = self.query(torch.tanh(self.proj(x))).squeeze(-1)
        a = a.masked_fill(~mask, -1e4)                # padding gets ~0 weight; -1e4 is safe in float16
        w = torch.softmax(a, dim=-1)
        return (w.unsqueeze(-1) * x).sum(1), w


def _safe_mask(mask):
    # Attention cannot run over a row with no real items, so unmask the first slot of empty rows.
    empty = ~mask.any(1)
    mask = mask.clone()
    mask[empty, 0] = True
    return mask, empty


class TitleEmbedding(nn.Module):
    def __init__(self, word_emb, d_model, dropout):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(torch.as_tensor(word_emb), freeze=False, padding_idx=0)
        self.proj = nn.Linear(word_emb.shape[1], d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, tokens):                        # tokens [N, L] word IDs
        real = (tokens > 0).unsqueeze(-1)
        avg = (self.emb(tokens) * real).sum(1) / real.sum(1).clamp_min(1)
        return self.proj(self.drop(avg))              # [N, d_model]


class EntityEmbedding(nn.Module):
    def __init__(self, ent_in_dim, n_types, type_dim, d_ent, dropout):
        super().__init__()
        self.type_emb = nn.Embedding(n_types + 1, type_dim, padding_idx=0)   # 0 = empty slot / unseen type
        self.proj = nn.Linear(ent_in_dim + type_dim + 2, d_ent)
        self.drop = nn.Dropout(dropout)

    def forward(self, vecs, types, conf, in_title):   # [N, E, 100], [N, E] x 3
        real = vecs.abs().sum(-1) > 0                 # empty slots point to the zero row
        x = torch.cat([vecs, self.type_emb(types), conf.unsqueeze(-1), in_title.unsqueeze(-1)], dim=-1)
        x = self.proj(self.drop(x))                   # [N, E, d_ent]
        w = conf.clamp_min(1e-3) * real               # confidence weights, 0 for empty slots
        w = w / w.sum(1, keepdim=True).clamp_min(1e-6)
        return (w.unsqueeze(-1).to(x.dtype) * x).sum(1)   # no entity -> zero vector


class UserEncoder(nn.Module):
    def __init__(self, d_model, n_heads, query_dim, dropout, use_lstm=True):
        super().__init__()
        self.use_lstm = use_lstm
        if use_lstm:
            self.lstm = nn.LSTM(d_model, d_model, batch_first=True)
        self.mha = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.att = AdditiveAttention(d_model, query_dim)
        self.drop = nn.Dropout(dropout)
        self.empty_user = nn.Parameter(torch.zeros(d_model))

    def forward(self, hist_vecs, hist_mask, return_weights=False):   # [B, H, d], [B, H] (real clicks first)
        mask, empty = _safe_mask(hist_mask)
        h = hist_vecs
        if self.use_lstm:
            lengths = mask.sum(1).cpu()
            packed = nn.utils.rnn.pack_padded_sequence(h, lengths, batch_first=True, enforce_sorted=False)
            out, _ = self.lstm(packed)
            h, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=hist_vecs.shape[1])
            h = h.to(hist_vecs.dtype)
        h, _ = self.mha(h, h, h, key_padding_mask=~mask, need_weights=False)
        u, w = self.att(self.drop(h), mask)
        u = torch.where(empty.unsqueeze(1), self.empty_user.to(u.dtype).expand_as(u), u)
        return (u, w) if return_weights else u


class KLAR(nn.Module):
    def __init__(self, word_emb, tokens, cfg, features):
        super().__init__()
        self.register_buffer("tokens", torch.as_tensor(tokens), persistent=False)
        self.title = TitleEmbedding(word_emb, cfg["d_model"], cfg["dropout"])
        self.use_cat, self.use_ent = cfg["use_category"], cfg["use_entities"]
        width = cfg["d_model"]
        if self.use_cat:
            self.register_buffer("cat_ids", torch.as_tensor(features["cat_ids"]), persistent=False)
            self.register_buffer("sub_ids", torch.as_tensor(features["sub_ids"]), persistent=False)
            self.cat_emb = nn.Embedding(features["n_cat"] + 1, cfg["cat_dim"], padding_idx=0)   # 0 = unseen
            self.sub_emb = nn.Embedding(features["n_sub"] + 1, cfg["cat_dim"], padding_idx=0)
            width += 2 * cfg["cat_dim"]
        if self.use_ent:
            ent = features["entities"]
            self.register_buffer("ent_table", torch.as_tensor(ent["ent_table"]), persistent=False)  # fixed TransE
            for k in ("ent_idx", "ent_type", "ent_conf", "ent_title"):
                self.register_buffer(k, torch.as_tensor(ent[k]), persistent=False)
            self.register_buffer("has_ent", torch.as_tensor(features["has_ent"]), persistent=False)
            self.entity = EntityEmbedding(ent["ent_table"].shape[1], ent["n_types"], cfg["ent_type_dim"],
                                          cfg["ent_dim"], cfg["dropout"])
            width += cfg["ent_dim"] + 1                                 # + has-entity flag
        self.feat_drop = nn.Dropout(cfg["dropout"])
        self.fuse = nn.Linear(width, cfg["d_model"])                    # 557 -> 256 with all inputs
        self.user_encoder = UserEncoder(cfg["d_model"], cfg["n_heads"], cfg["query_dim"], cfg["dropout"],
                                        use_lstm=cfg["use_lstm"])

    def encode_news(self, rows):                                        # [N] article rows -> [N, d_model]
        parts, side = [self.title(self.tokens[rows])], []
        if self.use_cat:
            side += [self.cat_emb(self.cat_ids[rows]), self.sub_emb(self.sub_ids[rows])]
        if self.use_ent:
            side.append(self.entity(self.ent_table[self.ent_idx[rows]], self.ent_type[rows],
                                    self.ent_conf[rows], self.ent_title[rows]))
        if side:
            parts.append(self.feat_drop(torch.cat([s.to(parts[0].dtype) for s in side], dim=-1)))
        if self.use_ent:
            parts.append(self.has_ent[rows].unsqueeze(-1).to(parts[0].dtype))
        return self.fuse(torch.cat(parts, dim=-1))

    def forward(self, hist_rows, cand_rows):                            # [B, H], [B, C] article rows
        B, H = hist_rows.shape
        uniq, inv = torch.unique(torch.cat([hist_rows.reshape(-1), cand_rows.reshape(-1)]), return_inverse=True)
        vecs = self.encode_news(uniq)                                   # each distinct article encoded once
        hist_vecs = vecs[inv[:B * H]].view(B, H, -1)
        cand_vecs = vecs[inv[B * H:]].view(B, cand_rows.shape[1], -1)
        user = self.user_encoder(hist_vecs, hist_rows > 0)
        return (cand_vecs * user.unsqueeze(1)).sum(-1)                  # raw scores [B, C]

_m = KLAR(WORD_EMB, TOKENS, CFG, FEATURES)
print(f"KLAR: {sum(p.numel() for p in _m.parameters()):,} parameters "
      f"(of which word vectors {_m.title.emb.weight.numel():,}); news representation input width "
      f"{_m.fuse.in_features} -> {_m.fuse.out_features}")
del _m

KLAR: 9,355,924 parameters (of which word vectors 8,266,200); news representation input width 557 -> 256


## 9. Scoring helpers: score every article shown

These functions take a trained model and score **every** article shown in **every** impression of a day (full slates, no sampling), which is how the model is evaluated.

To keep it fast:
1. every article's news representation is computed **once** (`encode_all_news`)
2. every impression's user vector is computed **once** (`user_vectors`)
3. each displayed article's score = dot product of the two (`score_candidates`)

The output has one row per displayed article, with columns `impression_id`, `news_id` and `score`, the shared format every model uses.

- **`with_scores`:** puts the scores next to the click labels so `utils/metrics.py` can evaluate them. It runs the same checks as `attach_scores` (one score per row, same order, no missing values) but skips the merge, so it doesn't use extra memory on the 14M-row test day.
- **`tune_small`:** a fixed random 20k Tune impressions, scored after every epoch to pick the best one.

In [10]:
@torch.no_grad()
def encode_all_news(model, batch=2048):
    # [n_articles + 1, d] news representations; row r belongs to news.loc[r - 1], row 0 is padding
    model.eval()
    out = []
    for s in range(0, model.tokens.shape[0], batch):
        rows = torch.arange(s, min(s + batch, model.tokens.shape[0]), device=DEVICE)
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            out.append(model.encode_news(rows).float())
    return torch.cat(out)

@torch.no_grad()
def user_vectors(model, news_vecs, H, batch=1024, return_weights=False):
    users, weights = [], []
    for s in range(0, len(H), batch):
        hist = torch.as_tensor(H[s:s + batch], device=DEVICE)
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            u, w = model.user_encoder(news_vecs[hist], hist > 0, return_weights=True)
        users.append(u.float()); weights.append(w.float())
    users = torch.cat(users)
    return (users, torch.cat(weights)) if return_weights else users

@torch.no_grad()
def score_candidates(model, cand, H, h_index):
    model.eval()
    news_vecs = encode_all_news(model)
    imp_codes, imp_ids = pd.factorize(cand["impression_id"])
    h_rows = h_index.reindex(imp_ids).to_numpy()
    if np.isnan(h_rows.astype(float)).any():
        raise ValueError("some candidate impressions have no history row")
    users = user_vectors(model, news_vecs, H[h_rows.astype(np.int64)])          # one per impression
    news_rows = np.array(NEWS_ROW.reindex(cand["news_id"].to_numpy()).fillna(0), dtype=np.int64)
    imp_codes = np.array(imp_codes, dtype=np.int64)       # writable copy for torch
    scores = np.empty(len(cand), dtype=np.float32)
    for s in range(0, len(cand), 200_000):
        u = users[torch.as_tensor(imp_codes[s:s + 200_000], device=DEVICE)]
        v = news_vecs[torch.as_tensor(news_rows[s:s + 200_000], device=DEVICE)]
        scores[s:s + 200_000] = (u * v).sum(-1).cpu().numpy()
    return pd.DataFrame({"impression_id": cand["impression_id"].to_numpy(),
                         "news_id": cand["news_id"].to_numpy(), "score": scores})

def with_scores(cand, s):
    assert len(s) == len(cand), "every candidate needs exactly one score"
    assert (s["impression_id"].to_numpy() == cand["impression_id"].to_numpy()).all(), "row order differs"
    assert (s["news_id"].to_numpy() == cand["news_id"].to_numpy()).all(), "row order differs"
    assert np.isfinite(s["score"].to_numpy()).all(), "NaN or infinite scores"
    out = cand[[c for c in ("impression_id", "label", "is_cold") if c in cand]].copy()
    out["score"] = s["score"].to_numpy()
    return out

def sample_impressions(cand, n, seed):
    ids = cand["impression_id"].drop_duplicates()
    return cand[cand["impression_id"].isin(ids.sample(n=min(n, len(ids)), random_state=seed))]

tune_small = sample_impressions(tune_cand, CFG["n_tune_imps"], CFG["seed"])   # fixed subset for epoch selection
print(f"Tune subset for epoch selection: {tune_small.impression_id.nunique():,} impressions")

Tune subset for epoch selection: 20,000 impressions


## 10. Training loop

This defines how the model is trained, in `train_model`. The same function is used for the main model and the ablations.

- **Loss:** for each example `[click, neg1, neg2, neg3, neg4]`, the model gives 5 scores. **Softmax cross-entropy** turns them into probabilities and penalises the model when the click (position 0) doesn't get the highest one.
- **Optimiser:** **Adam** with learning rate 1e-4, in mini-batches of 64 examples. Each step, backpropagation updates every learned part: word vectors, category and type embeddings, linear layers, LSTM and attention. The TransE vectors stay fixed.
- **Choosing the best epoch:** after each epoch, the model scores the fixed 20k Tune impressions (14 Nov, full slates). The epoch with the highest **Tune AUC** is kept, so the test day (15 Nov) is never used during training or selection.
- **Saving progress:** the model is saved to Drive after every epoch. With `RESUME = True`, an interrupted run continues from the last saved epoch instead of starting over.

In [11]:
def train_model(cfg, run_name, train_H, train_H_rows, train_C, log_every=500):
    seed_everything(cfg["seed"])
    model = KLAR(WORD_EMB, TOKENS, cfg, FEATURES).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    last_ckpt = os.path.join(RUN_DIR, f"{run_name}_last.pt")
    best_ckpt = os.path.join(RUN_DIR, f"{run_name}_best.pt")
    log, start_epoch, best_auc = [], 0, -1.0
    if RESUME and os.path.exists(last_ckpt):
        st = torch.load(last_ckpt, map_location=DEVICE)
        model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"])
        log, start_epoch, best_auc = st["log"], st["epoch"] + 1, st["best_auc"]
        print(f"Resumed {run_name} after epoch {start_epoch}")

    H_dev = torch.as_tensor(train_H, device=DEVICE)
    rows_dev = torch.as_tensor(train_H_rows, device=DEVICE)
    C_dev = torch.as_tensor(train_C, device=DEVICE)
    target = torch.zeros(cfg["batch_size"], dtype=torch.long, device=DEVICE)   # the click is column 0
    n, bs = len(C_dev), cfg["batch_size"]
    for epoch in range(start_epoch, cfg["epochs"]):
        model.train()
        t0, total, perm = time.time(), 0.0, torch.randperm(n, device=DEVICE)
        for b, s in enumerate(range(0, n, bs)):
            idx = perm[s:s + bs]
            with torch.autocast(DEVICE.type, enabled=USE_AMP):
                logits = model(H_dev[rows_dev[idx]], C_dev[idx])
                loss = F.cross_entropy(logits.float(), target[:len(idx)])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            total += loss.item() * len(idx)
            if log_every and b % log_every == 0:
                print(f"  epoch {epoch + 1} batch {b:>6}/{math.ceil(n / bs)} loss {loss.item():.4f}")
        tune_res = evaluate(with_scores(tune_small, score_candidates(model, tune_small, tune_H, tune_idx)), "score")
        row = {"epoch": epoch + 1, "train_loss": total / n, **{f"tune_{k}": v for k, v in tune_res.items()},
               "minutes": (time.time() - t0) / 60}
        log.append(row)
        print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})
        if tune_res["auc"] > best_auc:
            best_auc = tune_res["auc"]
            torch.save({"model": model.state_dict(), "cfg": cfg, "epoch": epoch + 1}, best_ckpt)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": epoch,
                    "log": log, "best_auc": best_auc}, last_ckpt)

    best = torch.load(best_ckpt, map_location=DEVICE)
    model.load_state_dict(best["model"])
    log = pd.DataFrame(log)
    log.to_csv(os.path.join(RUN_DIR, f"{run_name}_log.csv"), index=False)
    model.best_epoch = int(best["epoch"])
    print(f"{run_name}: using epoch {model.best_epoch} "
          f"(tune AUC {log.loc[log.epoch == model.best_epoch, 'tune_auc'].item():.4f})")
    return model, log

## 11. Train the final model (training days: 9–13 Nov)

This runs the training loop (section 10) on the 1.5M training examples for up to 6 epochs, then keeps the epoch with the best Tune AUC. Its settings and chosen epoch are saved to `klar_v1_config.json`.

If a run with this `RUN` name has already finished, the cell just loads the saved best model instead of training again.

In [12]:
model, train_log = train_model(CFG, RUN, fit_H, train_H_rows, train_C)
with open(os.path.join(RUN_DIR, f"{RUN}_config.json"), "w") as f:
    json.dump({**CFG, "best_epoch": model.best_epoch, "trained_on": "fit 9-13 Nov"}, f, indent=2)
train_log

  epoch 1 batch      0/23438 loss 1.6249
  epoch 1 batch    500/23438 loss 1.3431
  epoch 1 batch   1000/23438 loss 1.2962
  epoch 1 batch   1500/23438 loss 1.4532
  epoch 1 batch   2000/23438 loss 1.4211
  epoch 1 batch   2500/23438 loss 1.3563
  epoch 1 batch   3000/23438 loss 1.1907
  epoch 1 batch   3500/23438 loss 1.3281
  epoch 1 batch   4000/23438 loss 1.3374
  epoch 1 batch   4500/23438 loss 1.3484
  epoch 1 batch   5000/23438 loss 1.5062
  epoch 1 batch   5500/23438 loss 1.2584
  epoch 1 batch   6000/23438 loss 1.3453
  epoch 1 batch   6500/23438 loss 1.1262
  epoch 1 batch   7000/23438 loss 1.2921
  epoch 1 batch   7500/23438 loss 1.3293
  epoch 1 batch   8000/23438 loss 1.3288
  epoch 1 batch   8500/23438 loss 1.4670
  epoch 1 batch   9000/23438 loss 1.3004
  epoch 1 batch   9500/23438 loss 1.3282
  epoch 1 batch  10000/23438 loss 1.3232
  epoch 1 batch  10500/23438 loss 1.2926
  epoch 1 batch  11000/23438 loss 1.2372
  epoch 1 batch  11500/23438 loss 1.2540
  epoch 1 batch 

,epoch,train_loss,tune_auc,tune_mrr,tune_ndcg@5,tune_ndcg@10,tune_n_impressions,tune_n_skipped,minutes
0,1,1.316954,0.6624,0.3051,0.3291,0.3916,20000,0,8.640548
1,2,1.268813,0.6722,0.3104,0.3365,0.3987,20000,0,8.858325
2,3,1.248686,0.6730,0.3132,0.3403,0.4016,20000,0,8.764817
3,4,1.234761,0.6814,0.3161,0.3436,0.4059,20000,0,8.731943
4,5,1.223655,0.6832,0.3182,0.3468,0.4091,20000,0,8.894956
5,6,1.214315,0.6806,0.3148,0.3437,0.4051,20000,0,8.859287


## 12.  Results: train vs tune vs test, cold vs warm, and by history length

Every split is scored the same way: the final model (best epoch) ranks **every** article shown in each impression, and AUC, MRR and nDCG are computed per impression with the shared `utils/metrics.py`, then averaged.

**Overall table:**
- **Train:** 20k impressions from the training days (9–13 Nov), re-scored on their full slates. Compared with Tune and Test, this shows how much the model overfits.
- **Tune:** 14 Nov, every impression. This day was used to pick the best epoch.
- **Test:** 15 Nov (MIND's official dev split), every impression. Scored once, never used for any choice.

**On the test day, two breakdowns:**
- **Cold vs warm:** impressions are grouped by the articles the user clicked:
  - *cold*: every clicked article had fewer than 10 displays on earlier days
  - *warm*: every clicked article had 10 or more
  - *mixed*: some of each

  Each group is shown next to a **random ranking** of the same slates. Raw MRR and nDCG depend on slate length and number of clicks, so compare groups by AUC or by the lift over random.
- **By history length:** the user-side cold start, i.e. how well the model does for users with 0, 1–5, 6–20, 21–50 or 50+ past clicks. Users with no history all receive the learned "empty user" vector.



In [13]:
def full_slates_from_impressions(imp_df, impression_ids, n, seed):
    # Expand 'N1-0 N2-1 ...' slate strings of a sample of the given impressions into candidate rows.
    pool = imp_df[imp_df.impression_id.isin(impression_ids)].drop_duplicates("impression_id")
    sample = pool.sample(n=min(n, len(pool)), random_state=seed)
    rows = sample[["impression_id", "impressions"]].assign(cand=sample["impressions"].str.split()).explode("cand")
    nid_lab = rows["cand"].str.rsplit("-", n=1)
    return pd.DataFrame({"impression_id": rows["impression_id"].to_numpy(),
                         "news_id": nid_lab.str[0].to_numpy(),
                         "label": nid_lab.str[1].astype(int).to_numpy()})

def save_scores(s, name):
    s.to_parquet(os.path.join(SCORE_DIR, f"{name}.parquet"), index=False)
    print(f"saved {name}: {len(s):,} rows, columns {list(s.columns)}")

results, scored = {}, {}

# ---- train: training impressions on full slates ----
slates = pd.read_parquet(os.path.join(PROC, FILES["fit_imp"]), columns=[COLS["impression_id"], COLS["impressions"]])
slates.columns = ["impression_id", "impressions"]
tr = full_slates_from_impressions(slates, np.unique(train_imp_ids), CFG["n_train_slate_imps"], CFG["seed"])
del slates
tr_H, tr_idx = build_history(fit_imp, impression_ids=tr.impression_id.unique())
s = score_candidates(model, tr, tr_H, tr_idx)
save_scores(s, f"{PREFIX}_train_sample")
results["train (training impressions)"] = evaluate(with_scores(tr, s), "score")

# ---- tune and test ----
for split, cand, H, idx in [("tune", tune_cand, tune_H, tune_idx), ("eval", eval_cand, eval_H, eval_idx)]:
    s = score_candidates(model, cand, H, idx)
    save_scores(s, f"{PREFIX}_{split}")
    scored[split] = with_scores(cand, s)
    results[split] = evaluate(scored[split], "score")
    del s
del scored["tune"]
gc.collect()

overall = pd.DataFrame(results).T
overall.to_csv(os.path.join(RUN_DIR, f"{RUN}_overall.csv"))
print("\nOverall (full slates):")
display(overall)

def random_scores(df):
    return df.assign(score=np.random.default_rng(CFG["seed"]).random(len(df)))

def segment_table(test):
    rand = random_scores(test)
    m_seg = evaluate_by_segment(test, "score", "is_cold")
    r_seg = evaluate_by_segment(rand, "score", "is_cold")
    t = pd.DataFrame({
        "impressions": m_seg["n_impressions"],
        "AUC": m_seg["auc"], "AUC_random": r_seg["auc"],
        "MRR": m_seg["mrr"], "MRR_random": r_seg["mrr"],
        "nDCG@10": m_seg["ndcg@10"], "nDCG@10_random": r_seg["ndcg@10"],
    }).reindex(["all", "cold", "warm", "mixed"]).dropna(how="all")
    t["MRR_lift"] = (t["MRR"] - t["MRR_random"]).round(4)
    return t

HIST_BINS = [-1, 0, 5, 20, 50, np.inf]
HIST_LABELS = ["0 (no history)", "1-5", "6-20", "21-50", "over 50"]

def history_table(test, imp_df):
    n_hist = imp_df.drop_duplicates("impression_id").set_index("impression_id")["history"].map(lambda h: len(parse_history(h)))
    rows = []
    for df, name in [(test, "model"), (random_scores(test), "random")]:
        per = evaluate(df, "score", per_impression=True)
        per["history"] = pd.cut(per["impression_id"].map(n_hist), HIST_BINS, labels=HIST_LABELS)
        rows.append(per.groupby("history", observed=False)[["auc", "mrr"]].mean().add_suffix(f"_{name}"))
        if name == "model":
            counts = per.groupby("history", observed=False).size().rename("impressions")
    t = pd.concat([counts] + rows, axis=1)
    t["MRR_lift"] = t["mrr_model"] - t["mrr_random"]
    t["share_of_impressions"] = t["impressions"] / t["impressions"].sum()
    return t[["impressions", "share_of_impressions", "auc_model", "mrr_model", "mrr_random", "MRR_lift"]].round(4)

test = scored["eval"]
by_segment = segment_table(test)
by_segment.to_csv(os.path.join(RUN_DIR, f"{RUN}_eval_by_segment.csv"))
print("\nTest (15 Nov) by impression segment:")
display(by_segment)

by_history = history_table(test, eval_imp)
by_history.to_csv(os.path.join(RUN_DIR, f"{RUN}_eval_by_history.csv"))
print("\nTest (15 Nov) by the user's number of past clicks:")
display(by_history)

saved step4_klar_train_sample: 799,703 rows, columns ['impression_id', 'news_id', 'score']
saved step4_klar_tune: 17,400,106 rows, columns ['impression_id', 'news_id', 'score']
saved step4_klar_eval: 14,085,557 rows, columns ['impression_id', 'news_id', 'score']

Overall (full slates):


,auc,mrr,ndcg@5,ndcg@10,n_impressions,n_skipped
train (training impressions),0.7701,0.4013,0.4532,0.5096,20000.0,0.0
tune,0.6837,0.3180,0.3471,0.4080,431517.0,0.0
eval,0.6474,0.2987,0.3286,0.3950,376471.0,0.0



Test (15 Nov) by impression segment:


,impressions,AUC,AUC_random,MRR,MRR_random,nDCG@10,nDCG@10_random,MRR_lift
all,376471,0.6474,0.5008,0.2987,0.2189,0.3950,0.2866,0.0798
cold,77199,0.6068,0.5006,0.3885,0.2986,0.4789,0.3819,0.0899
warm,262807,0.6555,0.5010,0.2869,0.2102,0.3799,0.2734,0.0767
mixed,36465,0.6755,0.5002,0.1937,0.1128,0.3266,0.1796,0.0809



Test (15 Nov) by the user's number of past clicks:


,impressions,share_of_impressions,auc_model,mrr_model,mrr_random,MRR_lift
history,,,,,,
0 (no history),11270,0.0299,0.5570,0.2289,0.2181,0.0108
1-5,55262,0.1468,0.5948,0.3040,0.2565,0.0475
6-20,131368,0.3489,0.6398,0.3059,0.2289,0.0769
21-50,103557,0.2751,0.6724,0.3033,0.2063,0.0970
over 50,75014,0.1993,0.6785,0.2863,0.1911,0.0952


### What the results show

**1. Train vs tune vs test** (AUC 0.770 → 0.684 → 0.647)
- The model ranks clearly better than random on every split (random AUC = 0.50).
- **Train → Tune (−0.09):** the train impressions are ones the model learned from, so some gap is expected. Tune AUC peaked at epoch 5 and only dipped slightly at epoch 6, so this is not runaway overfitting.
- **Tune → Test (−0.04):** the test day is further from training, so more of its articles are new and user interests have drifted.

**2. Cold vs warm** (test day)
- KLAR beats random in every group, including **cold** impressions (AUC 0.607 vs 0.50), so it can rank articles that have barely been shown before.
- Cold AUC is lower than warm (0.607 vs 0.656), as expected: new articles only have their content to go on.
- The **lift over random** on cold impressions (MRR +0.090) is as large as on warm ones (+0.077), so the content-only design keeps its benefit for new articles.
- Raw MRR and nDCG look higher for cold only because cold slates are shorter, so they are not compared directly.

**3. By history length** (the user-side cold start)
- AUC rises with the number of past clicks: 0.557 (none) → 0.595 (1–5) → 0.640 (6–20) → 0.672 (21–50) → 0.679 (over 50).
- Users with **no history** (3% of impressions) get only a small lift: they all share the learned "empty user" vector, so there is nothing to personalise with.
- The gain levels off beyond about 20 clicks. Since the model reads at most the latest 50, users with more than 50 look much like those with 21–50.

## 13. Export the news representations(shared)

The final model's **news representation** of every article (the 256-number vector it compares with the user vector) is saved so step 5 can measure how similar two articles are when re-ranking for diversity.

| File (in `data/embeddings/`) | What it contains |
|---|---|
| `step4_klar_news_vectors.npy` | one row of 256 numbers per article (float32) |
| `step4_klar_news_ids.parquet` | the `news_id` of each row, in the same order |
| `step4_klar_news_vectors.json` | a short description: which model and epoch, size, and how the vectors were made |



In [14]:
vecs = encode_all_news(model).cpu().numpy()[1:].astype(np.float32)   # drop padding row 0
ids = news[["news_id"]].copy()
assert len(vecs) == len(ids) and ids.news_id.is_unique
np.save(os.path.join(EMB_DIR, f"{PREFIX}_news_vectors.npy"), vecs)
ids.to_parquet(os.path.join(EMB_DIR, f"{PREFIX}_news_ids.parquet"), index=False)
with open(os.path.join(EMB_DIR, f"{PREFIX}_news_vectors.json"), "w") as f:
    json.dump({"model": RUN, "epoch": model.best_epoch, "trained_on": "fit 9-13 Nov",
               "dim": int(vecs.shape[1]), "n_articles": int(len(vecs)), "normalised": False,
               "source": "KLAR news representation: averaged word2vec title words (fine-tuned), category and "
                         "category_subcategory embeddings, confidence-weighted TransE entity embedding, fused by a linear layer",
               "row_order": f"same as {PREFIX}_news_ids.parquet"}, f, indent=2)
norms = np.linalg.norm(vecs, axis=1)
print(f"Saved {vecs.shape[0]:,} news representations of size {vecs.shape[1]} to {EMB_DIR}")
print(f"Vector length: median {np.median(norms):.2f}, min {norms.min():.2f}, max {norms.max():.2f}")

Saved 104,151 news representations of size 256 to /content/drive/MyDrive/BT4222_Project/data/embeddings
Vector length: median 9.94, min 4.02, max 32.66


## 14. Ablations: does each part of KLAR help?

Each ablation trains a copy of KLAR with **one thing changed** and compares it with the unchanged model (**KLAR base**). Since only one thing differs, any change in results comes from that part.

**Table A: model parts**, checking whether each part is worth keeping

| Run | What is removed | What it tests |
|---|---|---|
| KLAR base | nothing | the reference |
| without LSTM | the LSTM; the user encoder keeps only self-attention + additive attention | does reading clicks **in order** help? |
| without entities | the entity embedding; articles use title + category only | do **knowledge-graph entities** help? |

**Table B: settings**, checking whether the defaults are sensible

| Run | What changes |
|---|---|
| 4 heads | 16 → 4 attention heads |
| dropout 0.1 | 0.2 → 0.1 |

**How the runs are kept fair and quick**
- Every run, including the base, trains on the **same 200k clicks for 2 epochs** with the same seed.Scores will be lower than the full model's, so compare runs with each other, not with section 12.
- All runs are compared on the **same 20k Tune impressions** (14 Nov). The test day is never used to choose between them.
- Finished runs reload from Drive instead of retraining.

**Is a difference real or just noise?**
Because every run ranks the same impressions, the notebook computes the difference from the base **impression by impression**. Its **standard error (SE)** shows how much that difference could vary by chance.
- **More than about 2 SE:** a real difference
- **Less than about 2 SE:** can't be told apart from noise

Example: −0.021 with SE 0.0034 is about 6 SE, so it's real.

AUC on **cold-click impressions** is also shown, since that is where content inputs like entities should matter most.

In [15]:
ABL_BUDGET = dict(n_train_clicks=200_000, epochs=2)

# name: (table, settings changed from KLAR)
ABLATIONS = {
    "KLAR base":        ("A", {}),
    "without LSTM":     ("A", {"use_lstm": False}),
    "without entities": ("A", {"use_entities": False}),
    "4 heads":          ("B", {"n_heads": 4}),
    "dropout 0.1":      ("B", {"dropout": 0.1}),
}

def tune_per_impression(m):
    # per-impression metrics on the fixed tune subset, with each impression's cold/warm segment
    df = with_scores(tune_small, score_candidates(m, tune_small, tune_H, tune_idx))
    per = evaluate(df, "score", per_impression=True).set_index("impression_id")
    per["segment"] = label_impression_segment(df, "is_cold").reindex(per.index)
    return per

def compare_to_base(per, base):
    # mean metrics + paired difference from the base on the same impressions, with its SE
    def paired(a, b):
        d = (a - b).dropna()
        return d.mean(), d.std(ddof=1) / np.sqrt(len(d))
    cold = per.segment == "cold"
    row = {"auc": per.auc.mean(), "mrr": per.mrr.mean(), "ndcg@10": per["ndcg@10"].mean(),
           "auc_cold": per.loc[cold, "auc"].mean(), "auc_warm": per.loc[per.segment == "warm", "auc"].mean()}
    row["AUC_vs_base"], row["SE"] = paired(per.auc, base.auc)
    row["cold_AUC_vs_base"], row["cold_SE"] = paired(per.auc[cold], base.auc[cold])
    return row

if RUN_ABLATIONS:
    # one shared reduced training set for every run
    abl_imp_ids, abl_C, abl_H, abl_H_rows = training_set(ABL_BUDGET["n_train_clicks"])
    abl_runs = {}
    for name, (table, change) in ABLATIONS.items():
        run = "klar_abl_" + re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
        m, _ = train_model(dict(CFG, **ABL_BUDGET, **change), run, abl_H, abl_H_rows, abl_C, log_every=0)
        abl_runs[name] = (table, m.best_epoch, tune_per_impression(m))
        del m
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    base_per = abl_runs["KLAR base"][2]
    for table in ("A", "B"):
        t = pd.DataFrame({name: {"best_epoch": ep, **compare_to_base(per, base_per)}
                          for name, (tb, ep, per) in abl_runs.items() if tb == table or name == "KLAR base"}).T
        t = t.astype(float).round(4).astype({"best_epoch": int})
        t.to_csv(os.path.join(RUN_DIR, f"ablation_table_{table}_tune.csv"))
        print(f"\nTable {table} ({'components' if table == 'A' else 'settings'}; base = KLAR base, "
              f"{int((base_per.segment == 'cold').sum()):,} cold-click impressions):")
        display(t)
    del abl_C, abl_H, abl_H_rows
    gc.collect()

{'epoch': 1, 'train_loss': 1.3865, 'tune_auc': 0.6519, 'tune_mrr': 0.291, 'tune_ndcg@5': 0.3126, 'tune_ndcg@10': 0.3744, 'tune_n_impressions': 20000, 'tune_n_skipped': 0, 'minutes': 1.1781}
{'epoch': 2, 'train_loss': 1.323, 'tune_auc': 0.654, 'tune_mrr': 0.2963, 'tune_ndcg@5': 0.3193, 'tune_ndcg@10': 0.381, 'tune_n_impressions': 20000, 'tune_n_skipped': 0, 'minutes': 1.1817}
klar_abl_klar_base: using epoch 2 (tune AUC 0.6540)
{'epoch': 1, 'train_loss': 1.4096, 'tune_auc': 0.6471, 'tune_mrr': 0.2879, 'tune_ndcg@5': 0.3088, 'tune_ndcg@10': 0.3707, 'tune_n_impressions': 20000, 'tune_n_skipped': 0, 'minutes': 0.7019}
{'epoch': 2, 'train_loss': 1.3387, 'tune_auc': 0.651, 'tune_mrr': 0.2938, 'tune_ndcg@5': 0.3169, 'tune_ndcg@10': 0.3781, 'tune_n_impressions': 20000, 'tune_n_skipped': 0, 'minutes': 0.6875}
klar_abl_without_lstm: using epoch 2 (tune AUC 0.6510)
{'epoch': 1, 'train_loss': 1.3887, 'tune_auc': 0.6502, 'tune_mrr': 0.2901, 'tune_ndcg@5': 0.3111, 'tune_ndcg@10': 0.3729, 'tune_n_impr

,best_epoch,auc,mrr,ndcg@10,auc_cold,auc_warm,AUC_vs_base,SE,cold_AUC_vs_base,cold_SE
KLAR base,2,0.6540,0.2963,0.3810,0.6195,0.6664,0.0000,0.0000,0.0000,0.0000
without LSTM,2,0.6510,0.2938,0.3781,0.6185,0.6628,-0.0030,0.0007,-0.0010,0.0021
without entities,2,0.6547,0.2991,0.3845,0.5984,0.6742,0.0007,0.0012,-0.0211,0.0034



Table B (settings; base = KLAR base, 4,590 cold-click impressions):


,best_epoch,auc,mrr,ndcg@10,auc_cold,auc_warm,AUC_vs_base,SE,cold_AUC_vs_base,cold_SE
KLAR base,2,0.6540,0.2963,0.3810,0.6195,0.6664,0.0000,0.0000,0.0000,0.0000
4 heads,2,0.6551,0.2950,0.3788,0.6152,0.6696,0.0011,0.0010,-0.0043,0.0027
dropout 0.1,2,0.6542,0.2953,0.3801,0.6183,0.6672,0.0002,0.0005,-0.0012,0.0013


### What the ablations show

Each row is compared with **KLAR base** (Tune AUC 0.654). A difference counts as real when it is larger than about 2 SE.

**Table A: model parts**

| Removed | Overall AUC vs base | Cold-click AUC vs base | Conclusion |
|---|---|---|---|
| LSTM | −0.0030 (≈ 4 SE) | −0.0010 (noise) | **The LSTM helps.** Reading clicks in order gives a small but real gain, mainly on warm impressions. |
| Entities | +0.0007 (noise) | **−0.0211 (≈ 6 SE)** | **Entities matter for new articles.** Overall AUC is unchanged, but cold-click AUC drops from 0.620 to 0.598 without them. |

The entities result is the key finding. Without entities, warm AUC rises slightly (0.666 → 0.674) but cold AUC falls clearly. For articles with plenty of click data, the title and category are enough. For new articles, knowing **who and what** the story is about, through the TransE knowledge-graph vectors, is what places them near articles the user liked. That supports keeping entities in a model built for cold start.

**Table B: settings**

| Changed | Overall AUC vs base | Cold-click AUC vs base | Conclusion |
|---|---|---|---|
| 4 heads (instead of 16) | +0.0011 (≈ 1 SE) | −0.0043 (≈ 1.6 SE) | No real difference, so 16 is kept |
| Dropout 0.1 (instead of 0.2) | +0.0002 (noise) | −0.0012 (noise) | No real difference, so 0.2 is kept |

**Caveats:** these runs use a reduced budget (200k clicks, 2 epochs, one seed), so they rank design choices rather than measure full-scale gains. The cold-click comparisons are based on 4,590 impressions, so their SEs are larger than the overall ones.

## 15. Outputs for the rest of the pipeline

| Output | File(s) | Used by |
|---|---|---|
| **Tune and test scores**: one score per article shown in each impression | `data/scores/step4_klar_tune.parquet`, `step4_klar_eval.parquet` | **Step 5** re-ranks for diversity (MMR), choosing λ on the Tune scores and applying it once to Test. **Step 6** compares models (diagnostics). |
| **Training-impression scores** | `data/scores/step4_klar_train_sample.parquet` | **Step 6**, for the train vs test (overfitting) comparison |
| **Article vectors** (news representations) | `data/embeddings/step4_klar_news_vectors.npy` + `step4_klar_news_ids.parquet` | **Step 5**, to measure how similar two articles are |
| **Saved models, training logs and result tables** | `outputs/step4_klar/` | The report |